# Cosmological Sachs Field Evolution

Full pipeline: FLRW cosmology (perFLRW) $\to$ source generation $\to$ 3-field Sachs evolution (sachsfield).

**Setup:** $z \in [0, 10]$, positive affine parameter $\lambda$ (past-directed geodesic, $\lambda$ increases with $z$), $n_{\rm fields} = 3$.

**Requirements:** `pyccl` environment (`conda run -n pyccl`).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import sachsfield as sf
from perFLRW import FLRWCosmology

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. Cosmology and Affine Parameter Grid

For a past-directed null geodesic, the affine parameter satisfies
$$\frac{d\lambda}{dz} = \frac{1}{(1+z)^2 H(z) E_{\rm co}}$$
with $\lambda(z=0) = 0$ and $\lambda > 0$ for $z > 0$.

Note: `FLRWCosmology` internally uses the *opposite* sign convention ($\lambda < 0$). We negate to get positive $\lambda$.

In [ ]:
flrw = FLRWCosmology(
    Omega_c=0.25, Omega_b=0.05, h=0.7, sigma8=0.8, n_s=0.96,
    E_co=1.0, z_max=10.0, n_table=2000,
)
print(flrw)

# Redshift grid: 2000 points from z=0 to z=10
z_grid = np.linspace(0, 10, 2000)

# Positive affine parameter: negate the internal convention
lam_pos = -flrw.lambda_of_z(z_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z_grid, lam_pos)
axes[0].set_xlabel('$z$')
axes[0].set_ylabel(r'$\lambda$ [Mpc]')
axes[0].set_title(r'Affine parameter $\lambda(z)$')

axes[1].plot(lam_pos, z_grid)
axes[1].set_xlabel(r'$\lambda$ [Mpc]')
axes[1].set_ylabel('$z$')
axes[1].set_title(r'Redshift $z(\lambda)$')
plt.tight_layout()
plt.show()

FLRWCosmology(Omega_c=0.2500, Omega_b=0.0500, h=0.700, sigma8=0.800, n_s=0.960, E_co=1.0)


IndexError: index 0 is out of bounds for axis 0 with size 0

## 2. Background Ricci Focusing $\Phi_{00}(z)$

$$\Phi_{00} = 4\pi G\,(\rho + P)_{\rm total}\,\left(\frac{E_{\rm co}}{a}\right)^2$$

This drives the saddle-point equation as $\bar{s}_1(\lambda)$.

In [ ]:
z_plot = z_grid[1:]  # skip z=0
phi00 = flrw.background_phi00(z_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(z_plot, phi00)
axes[0].set_xlabel('$z$')
axes[0].set_ylabel(r'$\Phi_{00}$ [1/Mpc$^2$]')
axes[0].set_title(r'Background $\Phi_{00}(z)$')

lam_plot = -flrw.lambda_of_z(z_plot)
axes[1].semilogy(lam_plot, phi00)
axes[1].set_xlabel(r'$\lambda$ [Mpc]')
axes[1].set_ylabel(r'$\Phi_{00}$ [1/Mpc$^2$]')
axes[1].set_title(r'Background $\Phi_{00}(\lambda)$')
plt.tight_layout()
plt.show()

print(f'Phi_00(z=1) = {flrw.background_phi00(1.0):.4e} 1/Mpc^2')
print(f'Phi_00(z=5) = {flrw.background_phi00(5.0):.4e} 1/Mpc^2')

## 3. Cross-Redshift $C_\ell$ Structure

We compute the full cross-spectrum $C_\ell^{\Phi_{00}}(z_i, z_j)$ to check whether different redshift shells are correlated.

**Result:** In Limber approximation, the matrix is sharply diagonal $-$ shells at different comoving distances share no correlated structure. This justifies using `corr_func=None` (uncorrelated shells) in the source generation.

In [ ]:
# Compute cross-spectrum on a coarse grid (~3-5 min)
z_cross = np.linspace(0.1, 5.0, 20)
ell_cross, cl_matrix = flrw.sachs_cls_cross(
    z_cross, lmax=200, delta_z=0.01, field='phi00'
)
print(f'Cross-spectrum shape: {cl_matrix.shape}')
print(f'ell range: [{ell_cross[0]:.0f}, {ell_cross[-1]:.0f}]')

In [ ]:
# Raw C_l(z_i, z_j) heatmaps
ell_targets = [10, 50, 100]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, ell_t in zip(axes, ell_targets):
    idx = np.argmin(np.abs(ell_cross - ell_t))
    mat = cl_matrix[:, :, idx]
    im = ax.pcolormesh(z_cross, z_cross, mat, cmap='inferno', shading='auto')
    ax.set_xlabel('$z_i$'); ax.set_ylabel('$z_j$')
    ax.set_title(rf'$C_{{\ell={int(ell_cross[idx])}}}^{{\Phi_{{00}}}}(z_i, z_j)$')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(r'Cross-redshift $C_\ell$ $-$ sharply diagonal (Limber)', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

# Correlation coefficient r(z_i, z_j)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, ell_t in zip(axes, ell_targets):
    idx = np.argmin(np.abs(ell_cross - ell_t))
    mat = cl_matrix[:, :, idx]
    diag = np.diag(mat)
    norm = np.sqrt(np.outer(diag, diag))
    norm[norm == 0] = 1.0
    r_mat = mat / norm
    im = ax.pcolormesh(z_cross, z_cross, r_mat, cmap='RdBu_r',
                       vmin=-0.1, vmax=1.0, shading='auto')
    ax.set_xlabel('$z_i$'); ax.set_ylabel('$z_j$')
    ax.set_title(rf'$r(\ell={int(ell_cross[idx])})$')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(r'Correlation coefficient $-$ confirms diagonal dominance', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

print('Conclusion: C_l(z_i, z_j) ~ 0 for z_i != z_j.')
print('Using corr_func=None (uncorrelated shells) is physically correct.')

## 4. Sign Convention Wrappers

`FLRWCosmology` returns $\lambda < 0$. We need $\lambda > 0$ (past-directed). All wrapper functions accept positive $\lambda$ and internally negate for perFLRW.

In [ ]:
nside = 32
n_fields = 3
lmax_cosmo = 3 * nside - 1  # = 95

# z -> positive lambda conversion helper
def z_of_lam(lam_pos):
    """Redshift from positive affine parameter."""
    return flrw.z_of_lambda(-np.asarray(lam_pos))

# Background source for saddle point (positive lambda convention)
_sbar_neg = flrw.sbar_func(n_fields=n_fields)
def sbar_pos(lam):
    """sbar(lam_positive) -> ndarray(3,). Phi_00 in component 0."""
    return _sbar_neg(-lam)

# Pre-compute power spectra on a z grid (~2 min)
z_cl_grid = np.linspace(0.05, 10.0, 80)
_cl_neg = flrw.cl_func(z_array=z_cl_grid, lmax=lmax_cosmo, delta_z=0.01, n_fields=n_fields)

def cl_pos(lam):
    """cl_func(lam_positive) -> (cl_11, cl_22, cl_33)."""
    return _cl_neg(-lam)

# Verify
lam_test = -float(flrw.lambda_of_z(1.0))  # positive lambda at z=1
sb = sbar_pos(lam_test)
print(f'sbar at z=1 (lam={lam_test:.1f}): Phi_00 = {sb[0]:.4e}, s_2 = {sb[1]:.2e}, s_3 = {sb[2]:.2e}')
print(f'  cross-check: flrw.background_phi00(1.0) = {flrw.background_phi00(1.0):.4e}')

cl11, cl22, cl33 = cl_pos(lam_test)
print(f'Power spectra at z=1: max(cl_11)={cl11.max():.2e}, max(cl_22)={cl22.max():.2e}')

## 5. Source Generation

Generate Gaussian random source fields on the sphere at 60 $\lambda$-samples. Since $C_\ell(z_i, z_j)$ is diagonal (Section 3), each shell is generated independently (`corr_func=None`).

In [ ]:
# Lambda range
lam_start = 1e-3   # small positive (z ~ 0)
lam_end = -float(flrw.lambda_of_z(10.0))   # large positive (z = 10)
print(f'lam_span = ({lam_start}, {lam_end:.2f}) Mpc')

# Source sample points (60 values, uniform in z -> non-uniform in lambda)
z_source_samples = np.linspace(0.05, 10.0, 60)
lam_source = -flrw.lambda_of_z(z_source_samples)

source = sf.FullSkySource(
    cl_func=cl_pos,
    lam_samples=lam_source,
    nside=nside,
    corr_func=None,  # justified by diagonal C_l(z,z') structure
    seed=42,
    n_fields=n_fields,
)
source.generate()
print(f'Generated {len(lam_source)} source maps at nside={nside} ({source.npix} pixels)')

## 6. Sachs Field Evolution

Evolve the 3-field system:
$$\dot{x}_a = F_{abc}\,x_b\,x_c + s_a$$

Saddle point uses background $\Phi_{00}$ as $\bar{s}_1(\lambda)$. Fluctuations use the stochastic source fields. Linear mode (`linear_only=True`) for the first run.

In [ ]:
solver = sf.SachsFieldSolver(
    source=source,
    lam_span=(lam_start, lam_end),
    sbar_func=sbar_pos,
    n_fields=n_fields,
    linear_only=True,
    saddle_kwargs=dict(rtol=1e-8, atol=1e-10),
    fluct_kwargs=dict(rtol=1e-5, atol=1e-7, method='RK45', max_step=500.0),
)

# Evaluate at specific redshifts
z_eval = np.array([0.01, 0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
t_eval = -flrw.lambda_of_z(z_eval)  # positive lambda values
print(f't_eval (lambda): {np.round(t_eval, 1)}')
print(f't_eval (z):      {z_eval}')

result = solver.solve(t_eval=t_eval)
print('\nEvolution complete.')
print(f'Saddle chi_1 at z=1: {result.saddle(t_eval[z_eval == 1.0][0])[0]:.6f}')
print(f'  reference 2/lam:   {2.0 / t_eval[z_eval == 1.0][0]:.6f}')

## 7. Saddle Point Diagnostics

The background solution $\chi_a(\lambda)$. At small $\lambda$ (low $z$), $\chi_1 \approx 2/\lambda$ (zero-source asymptotic). The Ricci focusing $\Phi_{00}$ modifies this at larger $\lambda$.

In [ ]:
# Evaluate saddle on a dense grid
lam_dense = np.linspace(lam_start + 0.1, lam_end, 500)
chi_dense = np.array([result.saddle(l) for l in lam_dense]).T  # (3, 500)
z_dense = z_of_lam(lam_dense)

labels = [r'$\chi_1$', r'$\chi_2$', r'$\chi_3$']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# vs lambda
for a in range(3):
    axes[0].plot(lam_dense, chi_dense[a], label=labels[a])
axes[0].plot(lam_dense, 2.0 / lam_dense, 'k--', alpha=0.4, label=r'$2/\lambda$')
axes[0].set_xlabel(r'$\lambda$ [Mpc]')
axes[0].set_ylabel(r'$\chi_a$')
axes[0].set_title('Saddle point vs $\\lambda$')
axes[0].legend()
axes[0].set_xlim(0, lam_end)

# vs redshift
for a in range(3):
    axes[1].plot(z_dense, chi_dense[a], label=labels[a])
axes[1].plot(z_dense, 2.0 / lam_dense, 'k--', alpha=0.4, label=r'$2/\lambda(z)$')
axes[1].set_xlabel('$z$')
axes[1].set_ylabel(r'$\chi_a$')
axes[1].set_title('Saddle point vs $z$')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Field Maps and Power Spectra

Visualize fluctuation maps and their angular power spectra at selected redshifts.

In [ ]:
# Summary at z=1 (find index)
idx_z1 = np.argmin(np.abs(z_eval - 1.0))
fig, stats_z1 = sf.summary_statistics(
    result, lam_index=idx_z1, mode='fullsky', nside=nside,
    show_saddle=False,
    suptitle=f'Fluctuations at z={z_eval[idx_z1]:.1f}',
)
plt.show()
for a in range(3):
    print(f'Field {a+1}: std={stats_z1["std"][a]:.3e}, skew={stats_z1["skew"][a]:.3f}')

In [ ]:
# Summary at z=5
idx_z5 = np.argmin(np.abs(z_eval - 5.0))
fig, stats_z5 = sf.summary_statistics(
    result, lam_index=idx_z5, mode='fullsky', nside=nside,
    show_saddle=False,
    suptitle=f'Fluctuations at z={z_eval[idx_z5]:.1f}',
)
plt.show()
for a in range(3):
    print(f'Field {a+1}: std={stats_z5["std"][a]:.3e}, skew={stats_z5["skew"][a]:.3f}')

In [ ]:
# Compare evolved power spectra vs input source spectra at z=1
xi_maps = result.fluctuation.xi[idx_z1]  # (3, npix)
ells_meas, cls_meas = sf.angular_power_spectrum(xi_maps, mode='fullsky', nside=nside)

# Input source C_l at the same lambda
cl_in = cl_pos(t_eval[idx_z1])
ell_in = np.arange(len(cl_in[0]))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['C0', 'C1', 'C2']
cl_labels_in = [r'$C_\ell^{11}$ (input)', r'$C_\ell^{22}$ (input)', r'$C_\ell^{33}$ (input)']
cl_labels_out = [r'$C_\ell^{11}$ (evolved)', r'$C_\ell^{22}$ (evolved)', r'$C_\ell^{33}$ (evolved)']
for a in range(3):
    mask_in = (ell_in > 1) & (cl_in[a] > 0)
    ax.loglog(ell_in[mask_in], cl_in[a][mask_in], '--', color=colors[a], alpha=0.5, label=cl_labels_in[a])
    mask_out = (ells_meas > 1) & (cls_meas[a] > 0)
    ax.loglog(ells_meas[mask_out], cls_meas[a][mask_out], '-', color=colors[a], label=cl_labels_out[a])
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell$')
ax.set_title(f'Input source vs evolved fluctuation spectra at z={z_eval[idx_z1]:.1f}')
ax.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

## 9. Evolution of Summary Statistics

Track how the RMS, skewness, and excess kurtosis of the fluctuation fields evolve over redshift.

In [ ]:
from scipy.stats import skew, kurtosis

n_times = len(t_eval)
rms_arr = np.zeros((3, n_times))
skew_arr = np.zeros((3, n_times))
kurt_arr = np.zeros((3, n_times))

for t in range(n_times):
    xi = result.fluctuation.xi[t]
    for a in range(3):
        rms_arr[a, t] = np.std(xi[a])
        skew_arr[a, t] = skew(xi[a])
        kurt_arr[a, t] = kurtosis(xi[a])

labels = [r'$x_1$', r'$x_2$', r'$x_3$']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for a in range(3):
    axes[0].plot(z_eval, rms_arr[a], 'o-', label=labels[a])
axes[0].set_xlabel('$z$'); axes[0].set_ylabel('RMS'); axes[0].set_title('RMS vs redshift')
axes[0].legend()

for a in range(3):
    axes[1].plot(z_eval, skew_arr[a], 'o-', label=labels[a])
axes[1].axhline(0, color='k', ls=':', alpha=0.3)
axes[1].set_xlabel('$z$'); axes[1].set_ylabel('Skewness'); axes[1].set_title('Skewness vs redshift')
axes[1].legend()

for a in range(3):
    axes[2].plot(z_eval, kurt_arr[a], 'o-', label=labels[a])
axes[2].axhline(0, color='k', ls=':', alpha=0.3)
axes[2].set_xlabel('$z$'); axes[2].set_ylabel('Excess kurtosis'); axes[2].set_title('Kurtosis vs redshift')
axes[2].legend()

plt.tight_layout()
plt.show()

## 10. Total Field at $z = 10$

The full solution $x_a = \chi_a + \xi_a$ at the final output redshift.

In [ ]:
total = result.total_field(-1)  # (3, npix) at z=10

fig = plt.figure(figsize=(18, 4))
titles = [r'$x_1$', r'$x_2$', r'$x_3$']
for a in range(3):
    hp.mollview(
        total[a],
        title=titles[a] + f' at $z={z_eval[-1]:.0f}$',
        sub=(1, 3, a + 1),
        fig=fig.number,
        cmap='RdBu_r',
        nest=False,
    )
plt.show()

# Statistics
chi_end = result.saddle(t_eval[-1])
print(f'Saddle point at z=10:  chi = [{chi_end[0]:.6f}, {chi_end[1]:.6f}, {chi_end[2]:.6f}]')
print(f'Reference 2/lam:       {2.0 / t_eval[-1]:.6f}')
print()
for a in range(3):
    print(f'Total x_{a+1}: mean={np.mean(total[a]):.4e}, '
          f'std={np.std(total[a]):.4e}, '
          f'min={np.min(total[a]):.4e}, max={np.max(total[a]):.4e}')

In [ ]:
# PDF evolution of fluctuation xi_1 across redshifts
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.cm.viridis
norm = plt.Normalize(vmin=z_eval.min(), vmax=z_eval.max())

for t_idx in range(len(z_eval)):
    z_val = z_eval[t_idx]
    xi1 = result.fluctuation.xi[t_idx, 0]  # xi_1, all pixels
    counts, bin_edges = np.histogram(xi1, bins=60, density=True)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    ax.plot(bin_centers, counts, color=cmap(norm(z_val)), lw=1.5)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('$z$')

ax.set_xlabel(r'$\xi_1$ pixel value')
ax.set_ylabel('PDF')
ax.set_title(r'Evolution of the $\xi_1$ pixel PDF with redshift')
plt.tight_layout()
plt.show()